<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_04_missing_condition_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · Notebook 04 — What Happens If You Forget One

**Paired with L7.2 · Fundamental PDEs**

This is the most important notebook in Ex_07.

You are going to solve the panel again, correctly in every respect except one:
the velocity condition is left out. The PDE is right. Both boundary conditions
are right. The displacement condition is right. Only $u_t(x,y,0)$ is missing.

The loss will converge beautifully. The answer will be nonsense. And nothing
in the training output will tell you.

## Why this works so cleanly here

The panel starts **flat**. So $u \equiv 0$ everywhere, for all time:

* satisfies $u_{tt} = c^2\nabla^2 u$ exactly — both sides are zero;
* satisfies $u = 0$ on all four edges exactly;
* satisfies $u(x,y,0) = 0$ exactly.

Every term you kept is satisfied by the trivial solution. The only thing ruling
it out is the term you dropped.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The under-determined problem

### Your turn

In [ ]:
# TODO 1 --- the loss WITHOUT the velocity term -------------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  mse(f) + w_b*mse(b) + w_u*mse(iu)          three terms: the iv term is deliberately absent
#   line 2  ->  make_loss_missing(model_missing, xyt_f, xyt_b, xyt_0, u0)
def wave_residual(model, xyt):                  # as in notebook 03
    u = model(xyt)
    return d2(u, xyt, 2) - pb.C_WAVE ** 2 * (d2(u, xyt, 0) + d2(u, xyt, 1))

U_SCALE = pb.V_STRIKE / pb.OMEGA
R_SCALE = U_SCALE * pb.OMEGA ** 2

def make_loss_missing(model, xyt_f, xyt_b, xyt_0, u0, w_b=1.0, w_u=1.0):
    def loss():
        f = wave_residual(model, xyt_f) / R_SCALE
        b = model(xyt_b) / U_SCALE
        iu = (model(xyt_0) - u0) / U_SCALE
        return ...                                # <- mse(f) + w_b*mse(b) + w_u*mse(iu)
    return loss

N_F, N_B, N_0 = 6000, 30, 800
T_SPAN = (0.0, pb.WAVE_T_END)
set_seed(88)
model_missing = MLP(n_in=3, n_hidden=48, n_layers=5)
xyt_f = to_tensor(spacetime_points(N_F, pb.WAVE_DOMAIN, T_SPAN, seed=1), requires_grad=True)
xyt_b = to_tensor(boundary_points_in_time(N_B, 24, pb.WAVE_DOMAIN, T_SPAN, seed=1))
init_np = initial_points(N_0, pb.WAVE_DOMAIN, t0=0.0, seed=1)
xyt_0 = to_tensor(init_np, requires_grad=True)
u0 = to_tensor(pb.wave_exact(init_np[:, 0], init_np[:, 1], 0.0).reshape(-1, 1))
v0 = to_tensor(pb.wave_velocity(init_np[:, 0], init_np[:, 1], 0.0).reshape(-1, 1))

history_missing = train_two_stage(model_missing, ...,        # <- make_loss_missing(model_missing, xyt_f, xyt_b, xyt_0, u0)
                                  adam_steps=6000, lbfgs_steps=250, lr=1e-3)
# ------------------------------------------------------------------------------

In [ ]:
plot_curves(history_missing, title="the panel, velocity condition omitted")
plt.show()

print(f"final loss, this notebook : {history_missing['lbfgs'][-1]:.3e}")
nb03 = np.load(os.path.join("Ex07.2_outputs", "nb03_panel.npz"))
print(f"final loss, notebook 03   : {nb03['lbfgs'][-1]:.3e}")

**What you should see.** A loss that falls further and faster than notebook
03's — often by several orders of magnitude.

Read that again. **The wrong problem converged better than the right one.**
Of course it did: it is an easier problem, and the function it is converging
to is the simplest one imaginable.

---

## 2 · Look at what it produced

### Your turn

In [ ]:
# TODO 2 --- how much does it move? --------------------------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  np.abs(centre_missing).max()
#   line 2  ->  np.abs(pb.wave_exact(q[:, 0], q[:, 1], ts)).max()
ts = np.linspace(0.0, pb.WAVE_T_END, 400)
mid = pb.L_PANEL / 2
q = np.stack([np.full_like(ts, mid), np.full_like(ts, mid), ts], axis=1)
with torch.no_grad():
    centre_missing = to_numpy(model_missing(to_tensor(q))).ravel()

peak_missing = ...                                # <- np.abs(centre_missing).max()
peak_exact   = ...                                # <- np.abs(pb.wave_exact(q[:, 0], q[:, 1], ts)).max()
fraction     = peak_missing / peak_exact
# ------------------------------------------------------------------------------

In [ ]:
pb.plot_time_history(
    ts,
    {"velocity condition omitted": centre_missing * 1e3,
     "both conditions (nb 03)": nb03["centre_pred"] * 1e3,
     "exact": nb03["centre_exact"] * 1e3},
    title="The panel that was struck, and the panel that was not",
    ylabel="centre displacement [mm]")
plt.show()

print(f"  peak deflection, exact           : {peak_exact*1e3:.4f} mm")
print(f"  peak deflection, model           : {peak_missing*1e3:.6f} mm")
print(f"  fraction of the real motion      : {fraction*100:.3f} %")
print()
print(f"  and the loss said                : {history_missing['lbfgs'][-1]:.3e}")

**What you should see.** A flat line at zero, against a sinusoid of amplitude
0.56 mm. The model captures well under a percent of the real motion, and often
far less.

**The panel was struck at half a metre per second and the model says it never
moved.**

---

## 3 · Every check you might have run, and what it says

### Your turn

In [ ]:
# TODO 3 --- the diagnostics a careful person would run ----------------------------------------------------
# Two `...` to replace (fresh points, never the training set):
#   line 1  ->  float(np.sqrt(np.mean(to_numpy(wave_residual(model_missing, xyt_new)) ** 2))) / R_SCALE
#   line 2  ->  float(np.abs(to_numpy(grad(u_new, xyt_0_new))[:, 2:3] - to_numpy(v0_new)).max())    the omitted one
xyt_new   = to_tensor(spacetime_points(2000, pb.WAVE_DOMAIN, T_SPAN, seed=9), requires_grad=True)
xyt_b_new = to_tensor(boundary_points_in_time(30, 12, pb.WAVE_DOMAIN, T_SPAN, seed=9))
init_new  = initial_points(500, pb.WAVE_DOMAIN, t0=0.0, seed=9)
xyt_0_new = to_tensor(init_new, requires_grad=True)
v0_new    = to_tensor(pb.wave_velocity(init_new[:, 0], init_new[:, 1], 0.0).reshape(-1, 1))

pde_res = ...                                     # <- float(np.sqrt(np.mean(to_numpy(wave_residual(model_missing, xyt_new)) ** 2))) / R_SCALE
with torch.no_grad():
    edge_err = float(np.abs(to_numpy(model_missing(xyt_b_new))).max())
u_new = model_missing(xyt_0_new)
ic_disp = float(np.abs(to_numpy(u_new)).max())
ic_vel  = ...                                     # <- float(np.abs(to_numpy(grad(u_new, xyt_0_new))[:, 2:3] - to_numpy(v0_new)).max())

diag = {"pde_res": pde_res, "edge_err": edge_err, "ic_disp": ic_disp, "ic_vel": ic_vel}
# ------------------------------------------------------------------------------

In [ ]:
print(error_table(
    [["PDE residual (RMS, scaled)", f"{diag['pde_res']:.3e}", "looks excellent"],
     ["edges, max |u|", f"{diag['edge_err']:.3e} m", "looks excellent"],
     ["u(x,y,0), max error", f"{diag['ic_disp']:.3e} m", "looks excellent"],
     ["u_t(x,y,0), max error", f"{diag['ic_vel']:.4f} m/s",
      "THE ONLY ONE THAT KNOWS"]],
    ["check", "value", "verdict"]))

**What you should see.** Three checks that pass magnificently, and one that
fails completely — the one you did not put in the loss.

This is the lesson of L7.2 and it generalises well beyond wave equations:

> **A converged residual tells you the network solves the problem you posed.
> It says nothing about whether you posed the right problem.**

In Ex_07.1 you could always compare against an exact solution. In L8 onward you
often cannot, and the residual is the only number you have. It is not enough.
Counting conditions — one per order in time, per independent variable — is a
five-second check that would have caught this.

---

## 4 · Two more ways to be under-determined

The missing initial condition is the cleanest example. It is not the only one.

### Your turn

In [ ]:
# TODO 4 --- one more way to be under-determined: a weakly sampled initial slice ---------------------------
# Two `...` to replace:
#   line 1  ->  20                      only twenty initial points instead of 800
#   line 2  ->  "..."                   your verdict, one sentence: where did the error appear, and did the loss say so?
def make_loss_full(model, xyt_f, xyt_b, xyt_0, u0, v0):
    def loss():
        f = wave_residual(model, xyt_f) / R_SCALE
        b = model(xyt_b) / U_SCALE
        u_pred = model(xyt_0)
        v_pred = grad(u_pred, xyt_0)[:, 2:3]
        return mse(f) + mse(b) + mse((u_pred - u0) / U_SCALE) + mse((v_pred - v0) / pb.V_STRIKE)
    return loss

init_few = initial_points(..., pb.WAVE_DOMAIN, t0=0.0, seed=1)    # <- 20
xyt_0_few = to_tensor(init_few, requires_grad=True)
u0_few = to_tensor(pb.wave_exact(init_few[:, 0], init_few[:, 1], 0.0).reshape(-1, 1))
v0_few = to_tensor(pb.wave_velocity(init_few[:, 0], init_few[:, 1], 0.0).reshape(-1, 1))

set_seed(88)
model_few = MLP(n_in=3, n_hidden=48, n_layers=5)
history_few = train_two_stage(model_few, make_loss_full(model_few, xyt_f, xyt_b, xyt_0_few, u0_few, v0_few),
                              adam_steps=6000, lbfgs_steps=250, lr=1e-3, report_every=0)
with torch.no_grad():
    centre_few = to_numpy(model_few(to_tensor(q))).ravel()
pb.plot_time_history(ts, {"20 initial points": centre_few * 1e3,
                          "800 initial points (nb 03)": nb03["centre_pred"] * 1e3,
                          "exact": nb03["centre_exact"] * 1e3},
                     title="Both conditions present, one of them barely sampled",
                     ylabel="centre displacement [mm]")
plt.show()
print(f"final loss with 20 initial points : {history_few['lbfgs'][-1]:.3e}   (nb 03: {nb03['lbfgs'][-1]:.3e})")
print(f"peak, 20 points : {np.abs(centre_few).max()*1e3:.4f} mm   exact {peak_exact*1e3:.4f} mm")

verdict = ...                                     # <- "write your one-sentence verdict here"
# ------------------------------------------------------------------------------

## 5 · Save

In [ ]:
path = os.path.join("Ex07.2_outputs", "nb04_missing_ic.npz")
np.savez(path,
         centre_missing=centre_missing, ts=ts,
         peak_missing=peak_missing, peak_exact=peak_exact, fraction=fraction,
         pde_res=diag["pde_res"], edge_err=diag["edge_err"],
         ic_disp=diag["ic_disp"], ic_vel=diag["ic_vel"],
         final_loss=history_missing["lbfgs"][-1],
         adam=history_missing["adam"], lbfgs=history_missing["lbfgs"])
print("wrote", path)

## 6 · Before you move on

1. State, in one sentence, why $u \equiv 0$ satisfied everything you kept.
2. The under-determined problem reached a **lower** loss than the correct one.
   Explain why that is not surprising, and why it is dangerous.
3. Give the counting rule for how many initial conditions a PDE needs, and
   apply it to the die and to the panel.
4. In L10 you will solve a battery model with no exact solution available. List
   two checks you could still run that would catch a mis-posed problem.

Next: **notebook 05**, the report.